In [2]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/sidecar_manager.py
import json
from pathlib import Path

class SidecarManager:
    def __init__(self, filepath="sidecar_edits.json"):
        # Inizializzazione di default; modificabile passandogli un diverso path come parametro
        self.filepath = Path(filepath) if filepath else Path("sidecar_edits.json")
        self._ensure_file_exists()

    def _ensure_file_exists(self):
        self.filepath.parent.mkdir(parents=True, exist_ok=True)
        if not self.filepath.exists():
            with open(self.filepath, "w", encoding="utf-8") as f:
                json.dump({
                    "pairwise_deltas": {},
                    "tag_overrides": {},
                    "global_tags": {}}, # Registro globale tag
                          f, indent=2)

    ### Accesso dati
    @property
    def data(self) -> dict:
        """Garantisce l'accesso diretto ai dati leggendoli sempre aggiornati dal file."""
        return self.load_data()

    def load_data(self) -> dict:
        if self.filepath.exists():
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return {"pairwise_deltas": {}, "tag_overrides": {}, "global_tags": {}}
        return {"pairwise_deltas": {}, "tag_overrides": {}, "global_tags": {}}

    def save_data(self, data: dict):
         with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    def get_global_tags(self) -> dict:
        """
        Restituisce il dizionario dei tag attivi, con contatore e colore corrente.
        """
        data = self.load_data()
        return data.get("global_tags", {})

    ### Manipolazione Distanze
    def save_pairwise_delta(self, chunk_id_1: str, chunk_id_2: str, distance_factor: float):
        """
        Salva o aggiorna il fattore di distanza tra una coppia di chunk.
        Garantisce la simmetria della relazione (A_B == B_A).
        """
        data = self.load_data()
        
        # Ordiniamo gli ID per garantire che la relazione sia simmetrica
        pair_key = "_AND_".join(sorted([str(chunk_id_1), str(chunk_id_2)]))
        
        if "pairwise_deltas" not in data:
            data["pairwise_deltas"] = {}

        data["pairwise_deltas"][pair_key] = {
            "chunk_1": str(chunk_id_1),
            "chunk_2": str(chunk_id_2),
            "distance_factor": round(distance_factor, 3)
        }

        self.save_data(data)
        print(f"<<| Modifica salvata per la coppia [{pair_key}]: factor={distance_factor:.2f} |>>")

    def save_pairwise_deltas_batch(self, edits: list):
        """
        Salva un blocco di modifiche pairwise in un unica operazione I/=
        """
        if not edits or not isinstance(edits, list):
            return

        data = self.load_data()
        if "pairwise_deltas" not in data:
            data["pairwise_deltas"] = {}

        updated = False
        for edit in edits:
            if edit and "chunk_1" in edit and "chunk_2" in edit:
                c1, c2 = str(edit["chunk_1"]), str(edit["chunk_2"])
                pair_key = "_AND_".join(sorted([c1, c2]))
                data["pairwise_deltas"][pair_key] = {
                    "chunk_1": c1,
                    "chunk_2": c2,
                    "distance_factor": round(float(edit["distance_factor"]), 3)
                }
                updated = True

        if updated:
            self.save_data(data)
            print(f"<<| Batch salvato: {len(edits)} distanze aggiornate nel sidecar |>>")

        
    ### Manipolazione TAG
    DEFAULT_PALETTE = [
        "#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f",
        "#edc949", "#af7aa1", "#ff9da7", "#9c755f", "#bab0ab"
    ] # Colori di Default se non ne vengono assegnati altri

    def add_tag_override(self, chunk_id: str, tag: str, color: str=None):
        """
        Aggiunge un tag a un chunk, assegna un colore e aggiorna il registro globale
        """
        data = self.load_data()
        chunk_id = str(chunk_id)

        if "tag_overrides" not in data:
            data["tag_overrides"] = {}
        if "global_tags" not in data:
            data["global_tags"] = {}

        if chunk_id not in data["tag_overrides"]:
            data["tag_overrides"][chunk_id] = {"user_tags": []}

        current_tags = data["tag_overrides"][chunk_id].get("user_tags", [])

        # Aggiunge il tag solo se non già presente
        if tag not in current_tags:
            current_tags.append(tag)
            data["tag_overrides"][chunk_id]["user_tags"] = current_tags

            # Aggiorna il registro globale
            if tag not in data["global_tags"]:
                # Se color=None si assegna un colore tra quelli di default
                if not color:
                    used_colors = {t_info.get("color") for t_info in data["global_tags"].values()}
                    available = [c for c in self.DEFAULT_PALETTE if c not in used_colors]
                    color = available[0] if available else self.DEFAULT_PALETTE[len(data["global_tags"]) % len(self.DEFAULT_PALETTE)] # Gestione ciclica array
                data["global_tags"][tag] = {"count": 1, "color": color}
            else:
                data["global_tags"][tag]["count"] += 1
                if color: # Aggiornamento colore se esplicitamente fornito
                    data["global_tags"][tag]["color"] = color

            self.save_data(data)
            print(f"<<! Tag '{tag}' ({data['global_tags'][tag]['color']}) aggiunto a [{chunk_id}]. Conteggio globale: {data['global_tags'][tag]['count']}")

    def remove_tag_override(self, chunk_id: str, tag: str):
        """
        Rimuove un tag da un chunk, decrementando il contatore globale.
        Se questo scende a 0, rimuove il tag da tale registro globale.
        """
        data = self.load_data()
        chunk_id = str(chunk_id)

        if "tag_overrides" in data and chunk_id in data["tag_overrides"]:
            current_tags = data["tag_overrides"][chunk_id].get("user_tags", [])
            if tag in current_tags:
                current_tags.remove(tag)
                data["tag_overrides"][chunk_id]["user_tags"] = current_tags

                # Rimuove la voce "tag_overrides" se non presente alcun tag
                if not current_tags:
                    del data["tag_overrides"][chunk_id]

                # Decrementa registro, rimuovendo la voce se necessario
                if "global_tags" in data and tag in data["global_tags"]:
                    data["global_tags"][tag]["count"] -= 1
                    if data["global_tags"][tag]["count"] == 0:
                        del data["global_tags"][tag]
                        print(f"X>> Tag '{tag}' rimosso definitivamente dal registro globale (conteggio = 0).")
                    else:
                        print(f"<<| Tag '{tag}' rimosso da [{chunk_id}]. Conteggio residuo: {data['global_tags'][tag]['count']}")
                
                self.save_data(data)
    
    ### RESET file sidecar
    def reset_all(self):
        """Ripristina il file sidecar azzerando le modifiche (UNDO globale)."""
        with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump({
                    "pairwise_deltas": {},
                    "tag_overrides": {},
                    "global_tags": {}},
                          f, indent=2)
        print("<<! File sidecar ripristinato allo stato iniziale !>>")
EOF

In [1]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/chunk_widget.py
import sys
import pathlib
from pathlib import Path
import anywidget
import traitlets
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import BASE_GRAPH_DISTANCE
from src.sidecar_manager import SidecarManager

class ChunkGraphWidget(anywidget.AnyWidget):
    _esm = pathlib.Path(__file__).parent / "frontend" / "chunk_graph.js"

    # Traitlets per stato del grafo
    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_tag = traitlets.Dict({}).tag(sync=True)
    global_tags = traitlets.Dict({}).tag(sync=True)
    tag_action = traitlets.Dict({}).tag(sync=True)

    #Traitlets per modifiche sulla distanza in batch -- assicura di non perdere edit
    pairwise_edit = traitlets.Dict({}).tag(sync=True)
    pairwise_edits_batch = traitlets.List([]).tag(sync=True)

    def __init__(self, sidecar_path=None, **kwargs):
        super().__init__(**kwargs)
        self.sidecar = SidecarManager(filepath=sidecar_path) if sidecar_path else SidecarManager()
        self.observe(self._on_pairwise_edit, names=["pairwise_edit"])
        self.observe(self._on_pairwise_edits_batch, names=["pairwise_edits_batch"])
        self.observe(self._on_tag_action, names=["tag_action"])
        self.refresh_global_tags()

    ### Gestione TAG
    def refresh_global_tags(self):
        """Sincronizza il dizionatio dei tag globali con il frontend"""
        self.global_tags = self.sidecar.get_global_tags()

    def _on_tag_action(self, change):
        """Gestisce le azioni di aggiunta e rimozione dei TAG inviate da JavaScript"""
        action_data = change["new"]
        if not action_data:
            return

        action = action_data.get("action")
        chunk_id = action_data.get("chunk_id")
        tag = action_data.get("tag")
        color = action_data.get("color")

        if action == "add" and chunk_id and tag:
            self.sidecar.add_tag_override(chunk_id, tag, color=color)
        elif action == "remove" and chunk_id and tag:
            self.sidecar.remove_tag_override(chunk_id, tag)

        self.refresh_global_tags()
        self._update_node_tags_in_graph_data(chunk_id)

    def _update_node_tags_in_graph_data(self, chunk_id):
        """
        Aggiorna i tag del nodo specifico in graph_data per forzare il ridisegno.
        Semplicemente ricrea e riposiziona i dati (uguali) nel nodo del chunk per forzare il re-render
        """
        data = self.sidecar.load_data()
        chunk_overrides = data.get("tag_overrides", {}).get(str(chunk_id), {})
        user_tags = chunk_overrides.get("user_tags", [])

        new_graph_data = dict(self.graph_data)
        nodes = [dict(n) for n in new_graph_data.get("nodes", [])]
        for n in nodes:
            if str(n.get("id")) == str(chunk_id):
                n["user_tags"] = user_tags
        new_graph_data["nodes"] = nodes
        self.graph_data = new_graph_data

    ### Caricamento grafo
    def load_graph(self, raw_graph_data):
        """Carica il grafo applicando immediatamente i distance_factor salvati nel sidecar."""
        data = self.sidecar.load_data()
        deltas = data.get("pairwise_deltas", {})
        overrides = data.get("tag_overrides", {})

        links = raw_graph_data.get("links", [])
        enriched_links = []

        # archi
        for link in links:
            l_copy = dict(link)
            src = l_copy["source"]["id"] if isinstance(l_copy["source"], dict) else l_copy["source"]
            tgt = l_copy["target"]["id"] if isinstance(l_copy["target"], dict) else l_copy["target"]

            k1 = f"{src}_AND_{tgt}"
            k2 = f"{tgt}_AND_{src}"

            factor = 1.0
            if k1 in deltas:
                factor = deltas[k1].get("distance_factor", 1.0)
            elif k2 in deltas:
                factor = deltas[k2].get("distance_factor", 1.0)

            l_copy["distance_factor"] = factor
            enriched_links.append(l_copy)

        # info sui nodi
        nodes = raw_graph_data.get("nodes", [])
        enriched_nodes = []
        for node in nodes:
            n_copy = dict(node)
            cid = str(n_copy.get("id"))
            if cid in overrides:
                n_copy["user_tags"] = overrides[cid].get("user_tags", [])
            elif "user_tags" not in n_copy:
                n_copy["user_tags"] = n_copy.get("tags", [])
            enriched_nodes.append(n_copy)
        
        self.graph_data = {
            "nodes": enriched_nodes,
            "links": enriched_links,
            "base_distance": BASE_GRAPH_DISTANCE
        }
        self.refresh_global_tags()

    ### Gestione edit Distanze
    def _on_pairwise_edit(self, change):
        """Gestisce una singola modifica di distanza inviata da JS -- versione meno sicura di pairwise_edits_batch"""
        edit = change["new"]
        if edit and "chunk_1" in edit and "chunk_2" in edit:
            self.sidecar.save_pairwise_delta(
                chunk_id_1=edit["chunk_1"],
                chunk_id_2=edit["chunk_2"],
                distance_factor=edit["distance_factor"]
            )

    def _on_pairwise_edits_batch(self, change):
        """
        Gestisce un blocco (tramite lista) di modifiche simultanee inviato in una sola volta.
        Si verifica quando il trascinamento di 1 nodo altera la distanza con più nodi ad esso collegati.
        Delegata al SidecarManager.
        """
        edits = change["new"]
        self.sidecar.save_pairwise_deltas_batch(edits)
EOF

In [8]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/graph_builder.py
import sys
from pathlib import Path
import networkx as nx
import numpy as np
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import SIMILARITY_THRESHOLD

class KnowledgeGraphBuilder:
    def __init__(self):
        self.graph = nx.Graph()

    def add_chunk_node(self, chunk_id: str, text: str, tags: list = None):
        self.graph.add_node(chunk_id, text=text, tags=tags or [])

    def add_relation(self, source_id: str, target_id: str, weight: float = 1.0):
        self.graph.add_edge(source_id, target_id, weight=weight)

    def auto_connect_nodes(self, records: list):
        """
        Calcola la Cosine Similarity tra tutte le coppie di record e
        crea un arco se la similarità supera la soglia di limite
        """
        def cosine_sim(v1, v2):
            norm1 = np.linalg.norm(v1)
            norm2 = np.linalg.norm(v2)
            if norm1 == 0 or norm2 == 0:
                return 0.0
            return float(np.dot(v1, v2) / (norm1 * norm2))

        n = len(records)
        for i in range(n):
            for j in range(i+1, n):
                v1, v2 = records[i].vector, records[j].vector
                sim = float(cosine_sim(v1,v2))

                if sim >= SIMILARITY_THRESHOLD:
                    self.add_relation(
                        source_id=str(records[i].id),
                        target_id=str(records[j].id),
                        weight=sim
                    )

    def to_json_data(self) -> dict:
        data = nx.node_link_data(self.graph)
        if "edges" in data and "links" not in data:
            data["links"] = data.pop("edges")
        return data
EOF

In [7]:
### Per popolare un grafo
"""
builder = KnowledgeGraphBuilder() 

#Aggiungi i nodi dai record estratti da Qdrant for rec in records:
builder.add_chunk_node(str(rec.id), text=rec.payload.get("text", ""))

#Archi generati e filtrati automaticamente in base a config.py
builder.auto_connect_nodes(records)

# Esporta per D3.js
graph_json = builder.to_json_data()
"""

'\nbuilder = KnowledgeGraphBuilder() \n\n#Aggiungi i nodi dai record estratti da Qdrant for rec in records:\nbuilder.add_chunk_node(str(rec.id), text=rec.payload.get("text", ""))\n\n#Archi generati e filtrati automaticamente in base a config.py\nbuilder.auto_connect_nodes(records)\n\n# Esporta per D3.js\ngraph_json = builder.to_json_data()\n'

In [2]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/frontend/chunk_graph.js
import * as d3 from "https://esm.sh/d3@7";

export function render({ model, el }) {
  el.innerHTML = "";

  function str(val) {
    return String(val);
  }

  let currentSimulation = null;
  let renderPending = false;
  let selectedNodeId = null;
  let currentNodes = [];

  // --- LAYOUT PRINCIPALE FLEXBOX ---
  const mainContainer = d3.select(el)
    .append("div")
    .style("display", "flex")
    .style("flex-direction", "row")
    .style("gap", "15px")
    .style("font-family", "system-ui, -apple-system, sans-serif")
    .style("color", "#1e293b");

  // --- COLONNA SINISTRA: GRAFO D3.js ---
  const graphContainer = mainContainer.append("div")
    .style("position", "relative")
    .style("width", "580px")
    .style("height", "420px");

  const width = 580;
  const height = 420;

  const svg = graphContainer.append("svg")
    .attr("width", width)
    .attr("height", height)
    .style("background", "#f8fafc")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "8px")
    .style("cursor", "grab");

  const defs = svg.append("defs");
  const g = svg.append("g");

  const zoom = d3.zoom()
    .scaleExtent([0.2, 4])
    .on("zoom", (event) => {
      g.attr("transform", event.transform);
    });

  svg.call(zoom);

  // --- COLONNA DESTRA: PANNELLO DI CONTROLLO ---
  const panel = mainContainer.append("div")
    .style("width", "320px")
    .style("height", "420px")
    .style("box-sizing", "border-box")
    .style("padding", "12px")
    .style("background", "#ffffff")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "8px")
    .style("display", "flex")
    .style("flex-direction", "column")
    .style("gap", "12px")
    .style("overflow-y", "auto");

  // Legenda
  const legendSection = panel.append("div")
    .style("padding-bottom", "10px")
    .style("border-bottom", "1px solid #e2e8f0");

  legendSection.append("div")
    .style("font-weight", "bold")
    .style("font-size", "13px")
    .style("margin-bottom", "6px")
    .text("*>> Tag Globali Attivi");

  const legendContainer = legendSection.append("div")
    .style("display", "flex")
    .style("flex-wrap", "wrap")
    .style("gap", "6px");

  // Dettagli Chunk
  const detailSection = panel.append("div")
    .style("display", "flex")
    .style("flex-direction", "column")
    .style("gap", "8px");

  detailSection.append("div")
    .style("font-weight", "bold")
    .style("font-size", "13px")
    .text(">> Chunk Selezionato");

  const detailContent = detailSection.append("div")
    .style("font-size", "12px")
    .style("color", "#64748b");

  const detailHeader = detailContent.append("div").style("font-weight", "bold").style("color", "#0f172a").style("margin-bottom", "4px");
  const detailText = detailContent.append("div")
    .style("max-height", "110px")
    .style("overflow-y", "auto")
    .style("padding", "6px 8px")
    .style("background", "#f1f5f9")
    .style("border-radius", "4px")
    .style("font-size", "11px")
    .style("margin-bottom", "8px");

  detailContent.append("div").style("font-weight", "600").style("font-size", "11px").style("margin-bottom", "4px").text("Tag Associati:");
  const currentTagsContainer = detailContent.append("div")
    .style("display", "flex")
    .style("flex-wrap", "wrap")
    .style("gap", "4px")
    .style("margin-bottom", "10px");

  const addTagBox = detailContent.append("div")
    .style("display", "flex")
    .style("gap", "4px");

  const inputTag = addTagBox.append("input")
    .attr("type", "text")
    .attr("placeholder", "Nuovo Tag...")
    .style("flex", "1")
    .style("padding", "4px 6px")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "4px")
    .style("font-size", "11px");

  const submitBtn = addTagBox.append("button")
    .text("+ Aggiungi")
    .style("padding", "4px 8px")
    .style("background", "#2563eb")
    .style("color", "#ffffff")
    .style("border", "none")
    .style("border-radius", "4px")
    .style("cursor", "pointer")
    .style("font-size", "11px");

  // Aggiunta Tag con controllo duplicati
  function submitTag() {
    const val = inputTag.property("value").trim();
    if (!val || !selectedNodeId) return;

    const graph = model.get("graph_data");
    const nodeData = graph && graph.nodes ? graph.nodes.find(n => str(n.id) === str(selectedNodeId)) : null;
    const userTags = nodeData ? (nodeData.user_tags || nodeData.tags || []) : [];

    if (!userTags.includes(val)) {
      model.set("tag_action", {
        action: "add",
        chunk_id: selectedNodeId,
        tag: val,
        timestamp: Date.now()
      });
      model.save_changes();
      inputTag.property("value", "");
    }
  }

  inputTag.on("keydown", (event) => { if (event.key === "Enter") submitTag(); });
  submitBtn.on("click", submitTag);

  function updateLegend() {
    const globalTags = model.get("global_tags") || {};
    legendContainer.html("");

    const tagNames = Object.keys(globalTags);
    if (tagNames.length === 0) {
      legendContainer.append("span").style("color", "#94a3b8").style("font-size", "11px").text("Nessun tag assegnato.");
      return;
    }

    tagNames.forEach(tagName => {
      const info = globalTags[tagName];
      legendContainer.append("span")
        .style("display", "inline-flex")
        .style("align-items", "center")
        .style("padding", "2px 8px")
        .style("border-radius", "12px")
        .style("font-size", "11px")
        .style("color", "#ffffff")
        .style("font-weight", "500")
        .style("background", info.color || "#6366f1")
        .text(`${tagName} (${info.count})`);
    });
  }

  // Generazione colore o gradiente per multi-tag
  function getNodeFill(d, globalTags) {
    const userTags = d.user_tags || d.tags || [];
    if (userTags.length === 0) return "#94a3b8";

    if (userTags.length === 1) {
      const tag = userTags[0];
      return (globalTags[tag] && globalTags[tag].color) ? globalTags[tag].color : "#6366f1";
    }

    // Nodi Multi-Tag: Creazione Gradiente SVG
    const gradId = `grad-${str(d.id).replace(/[^a-zA-Z0-9_-]/g, '_')}`;
    defs.select(`#${gradId}`).remove();

    const grad = defs.append("linearGradient")
      .attr("id", gradId)
      .attr("x1", "0%")
      .attr("y1", "0%")
      .attr("x2", "100%")
      .attr("y2", "100%");

    const count = userTags.length;
    userTags.forEach((t, i) => {
      const color = (globalTags[t] && globalTags[t].color) ? globalTags[t].color : "#6366f1";
      const startPercent = (i / count) * 100;
      const endPercent = ((i + 1) / count) * 100;

      grad.append("stop").attr("offset", `${startPercent}%`).attr("stop-color", color);
      grad.append("stop").attr("offset", `${endPercent}%`).attr("stop-color", color);
    });

    return `url(#${gradId})`;
  }

  function renderDetailPanel() {
    const graph = model.get("graph_data");
    const globalTags = model.get("global_tags") || {};

    if (!selectedNodeId || !graph || !graph.nodes) {
      detailHeader.text("");
      detailText.text("Clicca un nodo nel grafo per vederne il testo e gestirne i Tag.");
      currentTagsContainer.html("");
      addTagBox.style("display", "none");
      return;
    }

    const nodeData = graph.nodes.find(n => str(n.id) === str(selectedNodeId));
    if (!nodeData) {
      detailHeader.text("");
      detailText.text("Nodo non trovato.");
      currentTagsContainer.html("");
      addTagBox.style("display", "none");
      return;
    }

    addTagBox.style("display", "flex");
    detailHeader.text(`ID ${nodeData.id}`);
    detailText.text(nodeData.text || "Nessun testo associato.");

    currentTagsContainer.html("");
    const userTags = nodeData.user_tags || nodeData.tags || [];

    if (userTags.length === 0) {
      currentTagsContainer.append("span").style("font-size", "11px").style("color", "#94a3b8").text("Nessun tag.");
    } else {
      userTags.forEach(t => {
        const tagColor = (globalTags[t] && globalTags[t].color) ? globalTags[t].color : "#6366f1";
        const tagPill = currentTagsContainer.append("span")
          .style("display", "inline-flex")
          .style("align-items", "center")
          .style("gap", "4px")
          .style("padding", "2px 6px")
          .style("border-radius", "4px")
          .style("background", tagColor)
          .style("color", "#ffffff")
          .style("font-size", "11px");

        tagPill.append("span").text(t);
        tagPill.append("span")
          .style("cursor", "pointer")
          .style("font-weight", "bold")
          .style("margin-left", "2px")
          .text("✕")
          .on("click", () => {
            model.set("tag_action", {
              action: "remove",
              chunk_id: nodeData.id,
              tag: t,
              timestamp: Date.now()
            });
            model.save_changes();
          });
      });
    }
  }

  function draw() {
    if (currentSimulation) {
      currentSimulation.stop();
    }

    g.selectAll("*").remove();
    defs.selectAll("*").remove();

    const graph = model.get("graph_data");
    const globalTags = model.get("global_tags") || {};

    updateLegend();
    renderDetailPanel();

    if (!graph || !graph.nodes || graph.nodes.length === 0) return;

    const posMap = {};
    if (currentNodes) {
      currentNodes.forEach(n => {
        posMap[str(n.id)] = { x: n.x, y: n.y, fx: n.fx, fy: n.fy };
      });
    }

    const nodes = graph.nodes.map(d => {
      const existing = posMap[str(d.id)];
      if (existing) {
        return { ...d, x: existing.x, y: existing.y, fx: existing.fx, fy: existing.fy };
      }
      return { ...d };
    });

    currentNodes = nodes;
    const links = graph.links ? graph.links.map(d => ({ ...d })) : [];
    const BASE_DISTANCE = graph.base_distance || 120;

    const simulation = d3.forceSimulation(nodes)
      .force("link", d3.forceLink(links)
        .id(d => d.id)
        .distance(d => BASE_DISTANCE * (d.distance_factor || 1.0))
      )
      .force("charge", d3.forceManyBody().strength(-180))
      .force("center", d3.forceCenter(width / 2, height / 2));

    currentSimulation = simulation;

    const link = g.append("g")
      .selectAll("line")
      .data(links)
      .enter().append("line")
      .attr("stroke", "#94a3b8")
      .attr("stroke-width", 2);

    const linkText = g.append("g")
      .selectAll("text")
      .data(links)
      .enter().append("text")
      .attr("font-size", "11px")
      .attr("font-weight", "bold")
      .attr("fill", "#0284c7")
      .attr("text-anchor", "middle");

    const node = g.append("g")
      .selectAll("circle")
      .data(nodes)
      .enter().append("circle")
      .attr("r", 13)
      .attr("fill", d => getNodeFill(d, globalTags))
      .attr("stroke", d => str(d.id) === str(selectedNodeId) ? "#000000" : "#ffffff")
      .attr("stroke-width", d => str(d.id) === str(selectedNodeId) ? 3 : 2)
      .style("cursor", "grab");

    const label = g.append("g")
      .selectAll("text")
      .data(nodes)
      .enter().append("text")
      .text(d => d.id)
      .attr("font-size", "11px")
      .attr("dx", 16)
      .attr("dy", 4)
      .attr("fill", "#1e293b");

    simulation.on("end", () => {
      nodes.forEach(n => {
        n.fx = n.x;
        n.fy = n.y;
      });
    });

    // Binding corretto del drag rispetto allo zoom
    const drag = d3.drag()
      .container(g.node())
      .on("start", (event, d) => {
        nodes.forEach(n => {
          n.fx = n.x;
          n.fy = n.y;
        });
      })
      .on("drag", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;
        updatePositions();
      })
      .on("end", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;

        const batch = [];
        links.forEach(l => {
          const srcId = typeof l.source === 'object' ? l.source.id : l.source;
          const tgtId = typeof l.target === 'object' ? l.target.id : l.target;

          if (str(srcId) === str(d.id) || str(tgtId) === str(d.id)) {
            const dx = l.target.x - l.source.x;
            const dy = l.target.y - l.source.y;
            const currentDist = Math.hypot(dx, dy);
            const factor = currentDist / BASE_DISTANCE;

            l.distance_factor = factor;
            batch.push({
              chunk_1: srcId,
              chunk_2: tgtId,
              distance_factor: factor
            });
          }
        });

        if (batch.length > 0) {
          model.set("pairwise_edits_batch", batch);
          model.save_changes();
        }

        updatePositions();
      });

    node.call(drag);

    node.on("click", (event, d) => {
      if (str(selectedNodeId) !== str(d.id)) {
        inputTag.property("value", ""); // Reset input cambio nodo
      }

      selectedNodeId = d.id;
      renderDetailPanel();

      node.attr("stroke", n => str(n.id) === str(d.id) ? "#000000" : "#ffffff")
          .attr("stroke-width", n => str(n.id) === str(d.id) ? 3 : 2);

      model.set("selected_tag", { id: d.id, text: d.text || "" });
      model.save_changes();
    });

    function updatePositions() {
      link
        .attr("x1", d => d.source.x)
        .attr("y1", d => d.source.y)
        .attr("x2", d => d.target.x)
        .attr("y2", d => d.target.y);

      linkText
        .attr("x", d => (d.source.x + d.target.x) / 2)
        .attr("y", d => (d.source.y + d.target.y) / 2 - 5)
        .text(d => {
          const dx = d.target.x - d.source.x;
          const dy = d.target.y - d.source.y;
          const dist = Math.round(Math.hypot(dx, dy));
          const factor = (dist / BASE_DISTANCE).toFixed(2);
          return `${dist}px (${factor}x)`;
        });

      node.attr("cx", d => d.x).attr("cy", d => d.y);
      label.attr("x", d => d.x).attr("y", d => d.y);
    }

    simulation.on("tick", updatePositions);
    updatePositions(); // Posizionamento istantaneo
  }

  function scheduleDraw() {
    if (!renderPending) {
      renderPending = true;
      requestAnimationFrame(() => {
        renderPending = false;
        draw();
      });
    }
  }

  model.on("change:graph_data", scheduleDraw);
  model.on("change:global_tags", scheduleDraw);

  draw();

  return () => {
    if (currentSimulation) {
      currentSimulation.stop();
    }
    model.off("change:graph_data", scheduleDraw);
    model.off("change:global_tags", scheduleDraw);
  };
}
EOF